# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR^2) Exploration with `mlcroissant`
This notebook provides an interactive walkthrough for loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset uses a [Croissant schema](https://mlcommons.org/croissant/) for rich data interoperability and is openly available at:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the FAIR^2 dataset metadata and records using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata and records
dataset = mlc.Dataset(croissant_url)

# Display dataset metadata summary
meta = dataset.metadata
print(f"Dataset: {meta.name}\nDescription: {meta.description}\nVersion: {meta.version}\nIdentifier: {meta.identifier}")

## 2. Data Overview
Review available record sets, their `@id`s and included fields (columns). Croissant datasets organize data into record sets, each with fields that are also uniquely identified by `@id`.

In [ ]:
# List available record sets and their fields with @id
pp = pprint.PrettyPrinter(indent=2)
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in the metadata.")
else:
    for rs in record_sets:
        print(f"\nRecord Set: {rs.name} | @id: {rs.id}")
        try:
            fields = list(rs.fields)
            for field in fields:
                print(f"  Field: {field.name} | @id: {field.id} | type: {field.data_type}")
        except Exception as e:
            print("  (No fields or failed to enumerate fields)")

## 3. Data Extraction
Load all records from specific record sets into pandas DataFrames for analysis.

**Note:** All references are by `@id`. Use the printout above to pick a record set and its fields.

In [ ]:
# Collect all record set @ids
rs_ids = [rs.id for rs in record_sets]
print("Available record set @ids:", rs_ids)

# For demonstration, select the main tabular record set
# For this dataset, as of schema structure, assume first record set is the main data table
main_record_set_id = rs_ids[0] if rs_ids else None

dataframes = {}
if main_record_set_id is not None:
    for record_set_id in rs_ids:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Fields in record set '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets available for extraction.")

## 4. Exploratory Data Analysis (EDA)
Let's process and analyze the main record set. We'll demonstrate filtering, normalization and grouping using field `@id`s.

- **Filtering**: Select records where a numeric field meets a threshold.
- **Normalization**: Standardize a numeric column.
- **Grouping**: Group and aggregate data by a categorical field.

**Please update the field `@id`s below as listed earlier for your use case if desired.**

In [ ]:
# Select a numeric field @id (example: 'https://api.app.sen.science/frontiers/7862866/age_at_diagnosis')
# Replace below with the appropriate @id for e.g., 'Age at Diagnosis' column
numeric_field_id = None
group_field_id = None
main_df = dataframes[main_record_set_id]

# Heuristically search for 'age' or numeric field for demo
for col in main_df.columns:
    if 'age' in col.lower():
        numeric_field_id = col
        break
# Fallback: use first numeric column
if numeric_field_id is None:
    for col in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field_id = col
            break

# Heuristically set grouping field: try for 'sex', 'gender', or a short field
for col in main_df.columns:
    if 'sex' in col.lower() or 'gender' in col.lower():
        group_field_id = col
        break
if group_field_id is None and len(main_df.columns) > 1:
    group_field_id = main_df.columns[1]

print(f"Numeric field selected: {numeric_field_id}")
print(f"Grouping field selected: {group_field_id}")

if numeric_field_id is not None and numeric_field_id in main_df.columns:
    # Filtering example
    threshold = main_df[numeric_field_id].quantile(0.5) if pd.api.types.is_numeric_dtype(main_df[numeric_field_id]) else 10
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalization
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping 
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean')
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        display(grouped_df)
else:
    print("No suitable numeric field found for demonstration.")

## 5. Visualization
Visualize numeric field distributions and categorical groupings with matplotlib and seaborn.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in main_df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=10, kde=True, color='skyblue')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

    if group_field_id and group_field_id in main_df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id])
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field for plotting available.")

## 6. Conclusion
In this notebook, we loaded the FAIR^2 clinical dataset via Croissant, explored its record sets and field structure, extracted data by `@id`, and demonstrated basic EDA and visualization. The `mlcroissant` interface allows reproducible, schema-aware data exploration.

Further analysis can include:
- Statistical modeling of predictors,
- More detailed subgroup (e.g., MSI-H) analysis,
- Export of curated DataFrames for reuse.

Refer to the dataset's [Croissant schema documentation](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) for explicit `@id` references and field descriptions.